# TextSplitter的源码剖析

主要讲一下内部的三个方法（没有本质的区别，主要看参数类型和返回类型）：

方法1：
split_text(self, text: str) -> list[str]:
> 传入的参数类型：文本内容（或字符串），返回值类型：字符串列表
>
> 此方法是抽象方法，具体的实现细节由子类来决定

方法2：
create_documents(self, texts: list[str],...) -> list[Document]:
> 传入的参数类型：字符串列表，返回值类型：Document对象列表
>
> 此方法的底层调用了**split_text()**，即将参数中的每一个字符串都传入split_text()中执行，得到的字符串列表中，将字符串**封装**为Document对象，就构成了list[Document]。


方法3：
split_documents(self, documents: Iterable[Document]) -> list[Document]:
> 传入的参数类型：Document对象列表，返回值类型：Document对象列表
>
> 此方法的底层调用了**create_documents()**,将参数中的每一个Document对象，提取其page_content字段，则构成了字符串列表，然后调用方法2即可。

# 具体文档切分器的使用


## 1、CharacterTextSplitter：Split by character

举例1：字符串文本的分割

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text = """
Langchain是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，让开发者能够更容易地构建复杂地应用程序。
"""

# 定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size = 50,   # 每块大小，默认值为4000
    chunk_overlap = 5,    # 块与块之间地重复字符数
    # length_function = len,
    separator = ""   # 设置为空字符串时，表示禁用分隔符优先
)

# 分割文本
texts = splitter.split_text(text)

for i, chunk in enumerate(texts):
    print(f"块{i + 1}:长度:{len(chunk)}")
    print(chunk)
    print("-" * 50)

举例2：指定分割符

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分割基于字符数。"

# text = """
# LangChain 是一个用于开发由语言模型。驱动的应用程序的框架的。它提供了一套工具和抽象。使开发者能够更容易地构建复杂的应用程序。
# """

# 3.定义分割器实例
text_splitter = CharacterTextSplitter(
    chunk_size=30,   # 每个块的最大字符数
    chunk_overlap=5, # 块之间的重叠字符数
    separator="。",  # 按句号分割优先（注意）
)

# 4.开始分割
chunks = text_splitter.split_text(text)

# 5.打印效果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)


**separator优先原则**：当设置了 `separator`（如"。"），分割器会首先尝试在分隔符处分割，然后再考虑 chunk_size。这是为了避免在句子中间硬性切断。这种设计是为了：

1. 优先保持语义完整性（不切断句子）
2. 避免产生无意义的碎片（如半个单词/不完整句子）
3. 如果 `chunk_size` 比片段小，无法拆分片段，导致 overlap失效。
4. chunk_overlap仅在合并后的片段之间生效（如果 `chunk_size` 足够大）。如果没有合并的片段，则 overlap失效。

举例3：

In [5]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是第一段文本。这是第二段内容。最后一段结束。"

# 3.定义字符分割器
text_splitter = CharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=8,
    separator="。",
    # keep_separator=True #chunk中是否保留切割符
)

# 4.分割文本
chunks = text_splitter.split_text(text)

# 5.打印结果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)

块 1:长度：15
这是第一段文本。这是第二段内容
--------------------------------------------------
块 2:长度：14
这是第二段内容。最后一段结束
--------------------------------------------------
